# All Jobs for a User
Retrieve every job record for one Slurm username from the configured Elasticsearch indices. Set an optional submission or completion window to keep large result sets manageable.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import requests
import urllib3
from IPython.display import display

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

USERNAME = 'your_uniqname'
START = None  # e.g. '2026-09-01 00:00'; None = previous 24 hours
END = None  # None = now; end is exclusive
TIMEZONE = 'America/Detroit'
ES_URL = os.getenv('ES_URL', 'https://es.arc-ts.umich.edu')
ES_INDICES = 'slurm,slurm_greatlakes,slurm_lighthouse'
ES_VERIFY = os.getenv('ES_CA_BUNDLE') or False
PAGE_SIZE = 1000
EXPORT_DIR = Path('user_job_reports')

if not str(USERNAME).strip() or USERNAME == 'your_uniqname':
    raise ValueError('Set USERNAME before running the query.')

def date_window(start=None, end=None):
    def stamp(value):
        ts = pd.Timestamp(value)
        if ts.tzinfo is None:
            ts = ts.tz_localize(TIMEZONE)
        return ts.tz_convert('UTC')
    end = pd.Timestamp.now(tz='UTC') if end is None else stamp(end)
    start = end - pd.Timedelta(hours=24) if start is None else stamp(start)
    if start >= end:
        raise ValueError('START must be earlier than END.')
    return start, end

start, end = date_window(START, END)
filters = [
    {'term': {'username.keyword': USERNAME.strip()}},
    {'range': {'@submit': {'gte': start.isoformat(), 'lt': end.isoformat()}}}
]
query = {'query': {'bool': {'filter': filters}}, 'sort': ['_doc'],
         '_source': ['jobid', 'cluster', 'username', 'account', 'job_name', 'partition', 'state',
                     '@submit', '@start', '@end', 'elapsed', 'total_cpus', 'total_nodes',
                     'tres_alloc', 'array_job_id', 'array_task_id']}

es = requests.Session()
if os.getenv('ES_API_KEY'):
    es.headers['Authorization'] = 'ApiKey ' + os.environ['ES_API_KEY']
elif os.getenv('ES_USERNAME'):
    es.auth = (os.environ['ES_USERNAME'], os.environ['ES_PASSWORD'])

def es_request(method, path, **kwargs):
    response = es.request(method, ES_URL.rstrip('/') + path, timeout=60, verify=ES_VERIFY, **kwargs)
    response.raise_for_status()
    data = response.json()
    if data.get('timed_out') or data.get('_shards', {}).get('failed', 0):
        raise RuntimeError('Elasticsearch returned incomplete results; retry the query.')
    return data

In [ ]:
rows, scroll_id = [], None
try:
    page = es_request("POST", f"/{ES_INDICES}/_search", params={"scroll": "3m"},
                      json={**query, "size": PAGE_SIZE})
    while True:
        scroll_id = page.get("_scroll_id", scroll_id)
        hits = page["hits"]["hits"]
        if not hits:
            break
        rows.extend({**hit["_source"], "es_index": hit["_index"], "es_id": hit["_id"]} for hit in hits)
        page = es_request("POST", "/_search/scroll", json={"scroll": "3m", "scroll_id": scroll_id})
finally:
    if scroll_id:
        es_request("DELETE", "/_search/scroll", json={"scroll_id": [scroll_id]})

jobs = pd.DataFrame(rows)
if jobs.empty:
    print(f"No jobs found for {USERNAME!r}.")
else:
    for column in ["@submit", "@start", "@end"]:
        jobs[column] = pd.to_datetime(jobs.get(column), utc=True, errors="coerce")
    jobs["elapsed_seconds"] = pd.to_numeric(jobs.get("elapsed", 0), errors="coerce").fillna(0)
    jobs["runtime_hours"] = jobs["elapsed_seconds"] / 3600
    jobs = jobs.sort_values("@submit", ascending=False).reset_index(drop=True)
    print(f"Found {len(jobs):,} jobs for {USERNAME!r}.")
    display(jobs.head(50))
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    path = EXPORT_DIR / f"{USERNAME}_jobs_{pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%SZ')}.csv"
    jobs.to_csv(path, index=False)
    print(f"Exported results to {path}")